# Week 3 — Kampala Pilot + Controlled Runs (Augmentation On/Off)

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member:** AW
**Goal:** Add Kampala images, run controlled experiments:
- Run A: no augmentation (baseline)
- Run C: with augmentation (blur, brightness)

### RAM-safe design
- Same TinyCNN as Week 2
- 128×128, batch 8, 5 epochs
- Kampala images added as held-out test set
- Run cells one at a time

### Drive paths
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Kampala: `kampala/originals/`, `kampala/whatsapp_photo/`, `kampala/whatsapp_document/`
- Outputs: `reports/`

In [ ]:
!pip -q install torch torchvision scikit-learn pandas pillow

In [ ]:
import os, sys, json, glob, gc
from datetime import datetime
import numpy as np
import pandas as pd
from PIL import Image as PILImage, ImageFilter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)
print('Imports OK')

## 1. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'
MEMBER = 'AW'
N_IMAGES = 200
IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-3
SEED = 42
KAMPALA_MAX = 3

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

for d in ['reports', 'metadata', 'audit_outputs']:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

print('Drive root:', ROOT)
print('Kampala images to include:', KAMPALA_MAX)

## 2. Kampala log + image loader
Kampala images stored three ways per image ID: originals, whatsapp_photo, whatsapp_document.

In [ ]:
kampala_dirs = {
    'orig': os.path.join(ROOT, 'kampala', 'originals'),
    'wa_photo': os.path.join(ROOT, 'kampala', 'whatsapp_photo'),
    'wa_doc': os.path.join(ROOT, 'kampala', 'whatsapp_document'),
}
for d in kampala_dirs.values():
    os.makedirs(d, exist_ok=True)

kampala_log_path = os.path.join(ROOT, 'metadata', 'kampala_log.csv')

if not os.path.exists(kampala_log_path):
    template = pd.DataFrame(columns=['image_id', 'version', 'file_name', 'label', 'creator', 'device_or_tool', 'contains_person', 'consent_obtained'])
    template.to_csv(kampala_log_path, index=False)
    print('Created empty Kampala log template at', kampala_log_path)
else:
    print('Kampala log already exists at', kampala_log_path)

def load_kampala_images(kampala_dirs, max_ids, img_size):
    paths, labels, meta = [], [], []
    versions_per_id = {}
    log = pd.read_csv(kampala_log_path)
    if len(log) == 0:
        print('Kampala log is empty — no Kampala images to load.')
        return paths, labels, meta, versions_per_id
    ids = log['image_id'].unique()[:max_ids]
    for image_id in ids:
        rows = log[log['image_id'] == image_id]
        label = rows['label'].iloc[0]
        label_int = 0 if label == 'real' else 1
        versions = []
        for _, row in rows.iterrows():
            version = row['version']
            folder = kampala_dirs.get(version)
            if folder is None:
                continue
            fpath = os.path.join(folder, str(row['file_name']))
            if not os.path.exists(fpath):
                print(f'  WARNING: missing file {fpath}')
                continue
            paths.append(fpath)
            labels.append(label_int)
            meta.append({'path': fpath, 'label': label, 'source': 'kampala', 'image_id': image_id, 'version': version, 'creator': row.get('creator', MEMBER)})
            versions.append(version)
        versions_per_id[image_id] = versions
    print(f'Loaded {len(paths)} Kampala files across {len(ids)} image IDs')
    print(f'Versions per ID: {versions_per_id}')
    return paths, labels, meta, versions_per_id

kampala_paths, kampala_labels, kampala_meta, kampala_versions = load_kampala_images(kampala_dirs, KAMPALA_MAX, IMG_SIZE)

## 3. Load public pilot images + combine

In [ ]:
real_dir = os.path.join(ROOT, 'pilot_data', 'real')
ai_dir = os.path.join(ROOT, 'pilot_data', 'ai')

def load_public_images(real_dir, ai_dir, n_each, img_size):
    paths, labels, meta = [], [], []
    real_files = sorted(glob.glob(os.path.join(real_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(real_dir, '*.png')))
    for p in real_files[:n_each]:
        paths.append(p); labels.append(0)
        meta.append({'path': p, 'label': 'real', 'source': 'pilot'})
    ai_files = sorted(glob.glob(os.path.join(ai_dir, '*.jpg'))) + sorted(glob.glob(os.path.join(ai_dir, '*.png')))
    for p in ai_files[:n_each]:
        paths.append(p); labels.append(1)
        meta.append({'path': p, 'label': 'ai', 'source': 'pilot'})
    if len(paths) == 0:
        print('No pilot images — creating synthetic set.')
        for i in range(n_each):
            arr = np.random.randint(0, 255, (img_size, img_size, 3), dtype=np.uint8)
            im = PILImage.fromarray(arr)
            p = os.path.join(real_dir, f'synthetic_real_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(0)
            meta.append({'path': p, 'label': 'real', 'source': 'synthetic'})
            arr = np.zeros((img_size, img_size, 3), dtype=np.uint8)
            for x in range(img_size):
                arr[:, x, :] = [int(255 * x / img_size), 128, 64]
            im = PILImage.fromarray(arr)
            p = os.path.join(ai_dir, f'synthetic_ai_{i:03d}.jpg')
            im.save(p, quality=95)
            paths.append(p); labels.append(1)
            meta.append({'path': p, 'label': 'ai', 'source': 'synthetic'})
    return paths, labels, meta

n_each = N_IMAGES // 2
public_paths, public_labels, public_meta = load_public_images(real_dir, ai_dir, n_each, IMG_SIZE)

all_paths = public_paths + kampala_paths
all_labels = public_labels + kampala_labels
all_meta = public_meta + kampala_meta
print(f'Total images: {len(all_paths)} (public={len(public_paths)}, kampala={len(kampala_paths)})')
print(f'Label counts: real={sum(1 for l in all_labels if l==0)}, ai={sum(1 for l in all_labels if l==1)}')

## 4. Tensors + split (Kampala held out as test)

In [ ]:
to_tensor = transforms.ToTensor()
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def images_to_tensor(paths, labels, img_size):
    X_list, y_list = [], []
    for p, l in zip(paths, labels):
        im = PILImage.open(p).convert('RGB')
        t = normalize(to_tensor(im))
        X_list.append(t)
        y_list.append(float(l))
        im.close()
    return torch.stack(X_list), torch.tensor(y_list, dtype=torch.float32)

X_all, y_all = images_to_tensor(all_paths, all_labels, IMG_SIZE)

is_kampala = np.array([m['source'] == 'kampala' for m in all_meta])
public_idx = np.where(~is_kampala)[0]
kampala_idx = np.where(is_kampala)[0]

if len(public_idx) > 0:
    pub_X = X_all[public_idx]
    pub_y = y_all[public_idx]
    pub_meta = [all_meta[i] for i in public_idx]
    X_tr, X_va, y_tr, y_va, meta_tr, meta_va = train_test_split(pub_X, pub_y, pub_meta, test_size=0.2, random_state=SEED, stratify=pub_y.int().numpy())
else:
    X_tr, X_va, y_tr, y_va, meta_tr, meta_va = X_all[:0], X_all[:0], y_all[:0], y_all[:0], [], []

if len(kampala_idx) > 0:
    X_kamp = X_all[kampala_idx]
    y_kamp = y_all[kampala_idx]
    meta_kamp = [all_meta[i] for i in kampala_idx]
else:
    X_kamp, y_kamp, meta_kamp = X_all[:0], y_all[:0], []

train_ds = TensorDataset(X_tr, y_tr)
val_ds = TensorDataset(X_va, y_va)
kampala_ds = TensorDataset(X_kamp, y_kamp)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
kampala_loader = DataLoader(kampala_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

print(f'train={len(train_ds)} val={len(val_ds)} kampala_test={len(kampala_ds)}')
print(f'train labels: real={int((y_tr==0).sum())} ai={int((y_tr==1).sum())}')
if len(y_kamp) > 0:
    print(f'kampala labels: real={int((y_kamp==0).sum())} ai={int((y_kamp==1).sum())}')

## 5. TinyCNN model + training function

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, img_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x.squeeze(1)

def train_model(model, train_loader, val_loader, epochs, lr, augment=False, device=DEVICE):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    aug_transform = transforms.Compose([transforms.RandomHorizontalFlip(p=0.5), transforms.ColorJitter(brightness=0.2, contrast=0.2)]) if augment else None
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            if aug_transform is not None:
                xb_list = []
                for i in range(xb.size(0)):
                    img = transforms.ToPILImage()(xb[i].cpu())
                    img = aug_transform(img)
                    xb_list.append(transforms.ToTensor()(img))
                xb = torch.stack(xb_list).to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            n += xb.size(0)
        train_loss = total_loss / max(n, 1)
        model.eval()
        v_true, v_prob = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                probs = torch.sigmoid(model(xb)).cpu().numpy()
                v_prob.extend(probs.tolist())
                v_true.extend(yb.numpy().tolist())
        v_true = np.array(v_true).astype(int)
        v_prob = np.array(v_prob)
        v_pred = (v_prob >= 0.5).astype(int)
        v_acc = accuracy_score(v_true, v_pred) if len(v_true) > 0 else 0.0
        v_f1 = precision_recall_fscore_support(v_true, v_pred, average='macro', zero_division=0)[2] if len(v_true) > 0 else 0.0
        print(f'Epoch {epoch:2d} | train_loss={train_loss:.4f} | val_acc={v_acc:.4f} | val_f1={v_f1:.4f}')
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_acc': v_acc, 'val_f1': v_f1})
    gc.collect()
    return model, history

def evaluate_model(model, loader, device=DEVICE):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            probs = torch.sigmoid(model(xb)).cpu().numpy()
            y_prob.extend(probs.tolist())
            y_true.extend(yb.numpy().tolist())
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred) if len(y_true) > 0 else 0.0
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float('nan')
    cm = confusion_matrix(y_true, y_pred).tolist() if len(y_true) > 0 else [[0,0],[0,0]]
    return {'accuracy': float(acc), 'macro_f1': float(f1), 'precision': float(prec), 'recall': float(rec), 'roc_auc': float(auc), 'confusion_matrix': cm, 'n': int(len(y_true))}

## 6. Controlled Runs: Run A (no augmentation) vs Run C (augmentation)

In [ ]:
print('=' * 60)
print('RUN A: No augmentation (baseline)')
print('=' * 60)

model_a = TinyCNN(img_size=IMG_SIZE)
model_a, hist_a = train_model(model_a, train_loader, val_loader, EPOCHS, LR, augment=False, device=DEVICE)

res_a_val = evaluate_model(model_a, val_loader)
res_a_kampala = evaluate_model(model_a, kampala_loader) if len(kampala_ds) > 0 else {'n': 0}

print('\nRUN A results:')
print('  Val:', res_a_val)
print('  Kampala:', res_a_kampala)

torch.save(model_a.state_dict(), os.path.join(ROOT, 'reports', 'week3_run_a_no_aug.pt'))
gc.collect()

In [ ]:
print('=' * 60)
print('RUN C: With augmentation')
print('=' * 60)

model_c = TinyCNN(img_size=IMG_SIZE)
model_c, hist_c = train_model(model_c, train_loader, val_loader, EPOCHS, LR, augment=True, device=DEVICE)

res_c_val = evaluate_model(model_c, val_loader)
res_c_kampala = evaluate_model(model_c, kampala_loader) if len(kampala_ds) > 0 else {'n': 0}

print('\nRUN C results:')
print('  Val:', res_c_val)
print('  Kampala:', res_c_kampala)

torch.save(model_c.state_dict(), os.path.join(ROOT, 'reports', 'week3_run_c_aug.pt'))
gc.collect()

## 7. Compare Run A vs Run C + save results

In [ ]:
comparison = {
    'week': 3,
    'member': MEMBER,
    'model': 'TinyCNN',
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'kampala_image_ids_included': list(kampala_versions.keys()),
    'run_a_no_aug': {'val': res_a_val, 'kampala': res_a_kampala},
    'run_c_aug': {'val': res_c_val, 'kampala': res_c_kampala},
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}

comp_path = os.path.join(ROOT, 'reports', 'week3_augmentation_compare.json')
with open(comp_path, 'w') as f:
    json.dump(comparison, f, indent=2)

print('=== WEEK 3 COMPARISON ===')
print(f'Run A (no aug) val macro-F1: {res_a_val["macro_f1"]:.4f}')
print(f'Run C (aug)    val macro-F1: {res_c_val["macro_f1"]:.4f}')
if len(kampala_ds) > 0:
    print(f'Run A Kampala accuracy: {res_a_kampala["accuracy"]:.4f} (n={res_a_kampala["n"]})')
    print(f'Run C Kampala accuracy: {res_c_kampala["accuracy"]:.4f} (n={res_c_kampala["n"]})')
print(f'\nComparison saved to {comp_path}')
print('Models saved to reports/week3_run_a_no_aug.pt and week3_run_c_aug.pt')
gc.collect()